# DSC0152 - notebook v63

Target: KS OOF di atas `0.5299`, naik dari rekor sebelumnya `0.528717`.

## Yang berubah dari v62

1. **Feature selection jadi lever utama.** Sweep tiga fold seed (`42, 7, 13`) membandingkan level seleksi dan hyperparameter yang cocok untuknya. Tanpa seleksi KS 0.524794, dengan 200 kolom terpilih KS 0.527077 dengan sebaran antar seed paling kecil dari semua konfigurasi (sd 0.00017).
2. **Hipotesis awal salah, dan itu dicatat.** Dugaan bahwa `feature_fraction` 0.3 terlalu sempit untuk set 200 kolom ternyata tidak berlaku. Menaikkannya ke 0.6 menaikkan KS 0.0003 tetapi menurunkan AUC 0.0004 dan memperbesar sebaran antar seed tujuh kali, jadi 0.3 dipertahankan.
3. **Tangga seleksi sebagai sumbu ekstrapolasi baru.** Dua arah yang terbukti di leaderboard mendorong model berfitur lebih kaya menjauh dari model berfitur lebih sedikit. Ketajaman seleksi adalah ukuran lain dari quantity yang sama, jadi `sel200` didorong menjauh dari `sel400` memakai mekanisme yang persis sama.
4. **Base lintas run.** v62 dan v63 memakai empat fold seed yang sama, jadi prediksinya bisa dirata-ratakan sebelum diekstrapolasi. Karena kedua run menyimpan OOF per fold seed, pemeriksaan ketahanan bisa dilakukan di dalam tiap seed untuk semua kandidat lintas run.

## Hasil

| Kandidat | KS OOF | AUC | Korelasi vs jangkar |
|---|---:|---:|---:|
| `consensus5_best` | **0.529867** | 0.847204 | 0.995281 |
| `best_single` | 0.529767 | 0.847304 | 0.994953 |
| `consensus12` | 0.529620 | 0.847289 | 0.995430 |
| rekor sebelumnya (v62) | 0.528717 | 0.847128 | — |

`consensus5_best` naik `+0.001150` dari rekor sebelumnya. Per fold seed: `[0.52697, 0.52799, 0.52625, 0.52763]`, dan selisih terhadap base tanpa ekstrapolasi positif di 3 dari 4 seed dengan mean `+0.00089`.

## Batas yang jujur

Nilai `0.529867` masih `0.00013` di bawah `0.530000`. Pada 70.000 baris OOF, galat standar KS sekitar 0.002 sampai 0.003, jadi selisih sebesar itu tidak dapat dibedakan dari noise. Angka ini berarti mendekati `0.53` setelah pembulatan, bukan berarti melampaui `0.53`.

## Aturan yang dipatuhi

Python saja, hanya data kompetisi lokal, preprocessing dan feature engineering manual, tanpa AutoML dan AutoFE, tanpa data eksternal. Seleksi fitur dihitung per fold dari probe gain pada baris training fold itu saja. Factor loading dihitung dari label fold training saja. Semua agregat historis dipotong pada `decision_datetime`. Tidak ada push ke GitHub. Semua artefak hanya di `agas/`.

`RUN_TRAINING = False` membuat notebook hanya membaca cache sehingga bisa diverifikasi tanpa menghitung ulang.

In [ ]:
import os, sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd

RUN_TRAINING = False
FOLDS = 10
FOLD_SEEDS = '42,7,13,21'
LR = 0.01
L2 = 5.0
SEL200_SEEDS = '0,1'

ROOT = Path.cwd()
AGAS = ROOT / 'agas'
VER = AGAS / 'v63'
OUT = VER / 'output'
SHARED = AGAS / 'shared'
CACHE = SHARED / 'cache'
TRAINING = VER / 'training'
V62 = AGAS / 'v62' / 'output'
V61 = AGAS / 'v61' / 'output'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SHARED))
os.chdir(ROOT)
print('numpy', np.__version__, '| pandas', pd.__version__)
print('version dir', VER)

## 1. Data lokal

Tujuh tabel kompetisi dipakai. Tidak ada sumber eksternal.

In [ ]:
import lightgbm, xgboost, catboost, sklearn

print('lightgbm', lightgbm.__version__)
print('xgboost', xgboost.__version__)
print('catboost', catboost.__version__)
print('sklearn', sklearn.__version__)

apps = pd.read_csv(ROOT / 'data/train_applications.csv')
print('train rows', len(apps), '| base rate', round(apps.target.mean(), 5))

for name in ['train_applications', 'test_applications', 'customer_profiles',
             'previous_applications', 'repayment_history', 'account_behavior',
             'bureau_observations', 'ecommerce_transactions']:
    p = ROOT / 'data' / f'{name}.csv'
    print(f'{name:24s} {p.stat().st_size / 1e6:9.1f} MB')

## 2. Sweep level seleksi dan hyperparameter

Dua belas konfigurasi, masing-masing rata-rata tiga fold seed `42`, `7`, `13`. Seleksi dilakukan per fold dengan probe gain pada baris training fold tersebut.

In [ ]:
sweep = pd.read_csv(OUT / 'sweep_v63.json') if (OUT / 'sweep_v63.json').exists() else None
if sweep is None:
    sweep = json.load(open(SHARED / 'harness_output' / 'sweep_v63.json'))
    sweep = pd.DataFrame(sweep)
sweep_sorted = sweep.sort_values('ks', ascending=False)[['name', 'auc', 'ks', 'ks_sd', 'ks_per_seed']]
sweep_sorted

In [ ]:
if RUN_TRAINING:
    os.system(f'python3 {VER / "sweep_v63.py"}')

print('konfigurasi terbaik menurut KS:')
print(sweep_sorted.head(6).to_string(index=False))

## 3. Ensemble

10 fold, empat fold seed, dan kaki-kaki berikut. `selN` berarti N kolom terpilih gain per fold. `feature_fraction` mengikuti hasil sweep: 0.9 untuk 100 kolom, 0.3 untuk 200, 0.5 untuk 300, 0.4 untuk 400.

In [ ]:
cmd = (f'python3 {TRAINING / "train_submission_v63.py"} --folds {FOLDS} '
       f'--fold-seeds {FOLD_SEEDS} --lr {LR} --l2 {L2} '
       f'--sel200-seeds {SEL200_SEEDS} --out-prefix v63')
print(cmd)
if RUN_TRAINING:
    os.system(cmd)
else:
    print('dilewati, memakai agas/v63/output/preds_v63.npz yang sudah ada')

In [ ]:
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score

z = np.load(OUT / 'preds_v63.npz', allow_pickle=True)
seeds = [int(s) for s in z['seeds']]
train_ids, test_ids = z['train_ids'], z['test_ids']
ymap = apps.set_index('application_id').reindex(pd.Index(train_ids))
y = ymap['target'].values.astype(int)

rows = []
for k in sorted(c for c in z.files if c.startswith('oof_')):
    name = k[4:]
    p = z[k]
    per = [ks_2samp(z[f'oofseed_{name}_{j}'][y == 1], z[f'oofseed_{name}_{j}'][y == 0]).statistic
           for j in range(len(seeds))]
    rows.append({'model': name, 'auc': roc_auc_score(y, p),
                 'ks': ks_2samp(p[y == 1], p[y == 0]).statistic,
                 'ks_per_seed': [round(v, 5) for v in per]})
base = pd.DataFrame(rows).sort_values('ks', ascending=False).reset_index(drop=True)
base

## 4. Pencarian arah dan pemilihan consensus

Grid 1296 baris: empat base, delapan kaki, sembilan rasio untuk sumbu tunggal, dan semua pasangan kaki dengan tiga nilai per sumbu untuk dua sumbu. Arah hanya direkomendasikan bila KS membaik di sedikitnya tiga dari empat fold seed.

In [ ]:
g = pd.read_csv(OUT / 'gradient_v63_multirun.csv')
print(f"grid {len(g)} baris | best single {g.ks.max():.6f} | "
      f"robust {int(g.robust.sum())} ({100 * g.robust.mean():.1f}%)")
pd.set_option('display.width', 220)
g.sort_values('ks', ascending=False).head(12).to_string(index=False)

In [ ]:
if RUN_TRAINING:
    os.system(f'python3 {TRAINING / "search_multirun_v63.py"}')
    os.system(f'python3 {TRAINING / "final_v63.py"}')

summary = json.load(open(OUT / 'summary_v63_final.json'))
print('ukuran consensus terbaik:', summary['best_consensus_size'])
print('KS per fold seed, consensus:', summary['consensus_per_seed'])
print('KS per fold seed, base    :', [round(v, 5) for v in summary['base_per_seed']])
print('selisih per fold seed     :', [round(v, 5) for v in summary['delta_per_seed']])
pd.DataFrame(summary['size_sweep']).sort_values('ks', ascending=False).to_string(index=False)

In [ ]:
summary = json.load(open(OUT / 'summary_v63_final.json'))
cands = pd.DataFrame([{k: v for k, v in c.items() if k != 'problems'}
                     for c in summary['candidates']])
cands

## 5. Verifikasi tahan banting

`verify_v63_consensus.py` membangun ulang consensus dari kode yang sama lalu membandingkannya dengan file yang sudah ditulis. Kesamaan harus persis, supaya angka di laporan ini benar-benar berasal dari file submission.

In [ ]:
os.system(f'python3 {TRAINING / "verify_v63_consensus.py"}')

## 6. Validasi file submission

Setiap kandidat diperiksa ulang: 30000 baris, `application_id` unik dan sama dengan `data/test_applications.csv`, probabilitas di rentang [0, 1] tanpa NaN.

In [ ]:
ref = set(pd.read_csv(ROOT / 'data/test_applications.csv', usecols=['application_id']).application_id)

rows = []
for path in sorted(OUT.glob('submission_*.csv')):
    d = pd.read_csv(path)
    ok = (len(d) == 30000 and d.application_id.is_unique and set(d.application_id) == ref
          and d.default_probability.between(0, 1).all() and not d.default_probability.isna().any())
    rows.append({'file': path.name, 'rows': len(d), 'ok': ok,
                 'min': round(d.default_probability.min(), 6),
                 'max': round(d.default_probability.max(), 6)})
res = pd.DataFrame(rows)
print('total file:', len(res), '| invalid:', int((~res.ok).sum()))
res

## 7. Urutan submit

Jangkar tetap `submission_v61_alt_ext0.5_tm30_lean.csv` dengan public `0.56312`.

1. `submission_v63f_consensus5_best.csv`, KS OOF 0.529867, korelasi 0.995281 terhadap jangkar.
2. `submission_v63f_best_single.csv`, KS OOF 0.529767, korelasi 0.994953, dan profil per seed paling seragam.
3. `submission_v63f_consensus12.csv`, KS OOF 0.529620, rata-rata per seed tertinggi sehingga paling stabil.
4. `submission_v63d_cand02.csv` sampai `cand05.csv`, untuk menguji arah yang berbeda.
5. `submission_v63f_anchor35_consensus.csv`, 35 persen consensus baru di atas jangkar, untuk slot yang harus aman.

Korelasi terhadap jangkar sekitar 0.995, cukup jauh dari 0.9996 milik kandidat v62, jadi kandidat ini benar-benar menguji hipotesis baru dan bukan mengulang skor yang sudah diketahui.